In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('housing_price_dataset.csv')
df.sample(5)

,SquareFeet,Bedrooms,Bathrooms,Neighborhood,YearBuilt,Price
21656,1737,3,1,Rural,2017,238410.718960
16166,2205,3,2,Suburb,1961,239974.845534
746,2516,4,3,Suburb,1985,275162.445030
26136,2235,2,2,Suburb,2014,272761.655453
31166,1225,2,1,Urban,2004,248493.770211


In [4]:
# Check for missing values
df.describe()

,SquareFeet,Bedrooms,Bathrooms,YearBuilt,Price
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,2006.374680,3.498700,1.995420,1985.404420,224827.325151
std,575.513241,1.116326,0.815851,20.719377,76141.842966
min,1000.000000,2.000000,1.000000,1950.000000,-36588.165397
25%,1513.000000,3.000000,1.000000,1967.000000,169955.860225
50%,2007.000000,3.000000,2.000000,1985.000000,225052.141166
75%,2506.000000,4.000000,3.000000,2003.000000,279373.630052
max,2999.000000,5.000000,3.000000,2021.000000,492195.259972


In [5]:
# Check for missing values
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   SquareFeet    50000 non-null  int64  
 1   Bedrooms      50000 non-null  int64  
 2   Bathrooms     50000 non-null  int64  
 3   Neighborhood  50000 non-null  str    
 4   YearBuilt     50000 non-null  int64  
 5   Price         50000 non-null  float64
dtypes: float64(1), int64(4), str(1)
memory usage: 2.3 MB


In [6]:
df.isnull().sum() # check for missing values


SquareFeet      0
Bedrooms        0
Bathrooms       0
Neighborhood    0
YearBuilt       0
Price           0
dtype: int64

In [7]:
df['Neighborhood'].value_counts() # check for categorical values

Neighborhood
Suburb    16721
Rural     16676
Urban     16603
Name: count, dtype: int64

## Extracting the features

In [8]:
#drop the 'year' column
df.drop(columns=['YearBuilt'], inplace=True)

In [9]:
x = df.drop(columns=['Price'])
y = df['Price']

print(x)
print(y)

       SquareFeet  Bedrooms  Bathrooms Neighborhood
0            2126         4          1        Rural
1            2459         3          2        Rural
2            1860         2          1       Suburb
3            2294         2          1        Urban
4            2130         5          2       Suburb
...           ...       ...        ...          ...
49995        1282         5          3        Rural
49996        2854         2          2       Suburb
49997        2979         5          3       Suburb
49998        2596         5          2        Rural
49999        1572         5          3        Rural

[50000 rows x 4 columns]
0        215355.283618
1        195014.221626
2        306891.012076
3        206786.787153
4        272436.239065
             ...      
49995    100080.865895
49996    374507.656727
49997    384110.555590
49998    380512.685957
49999    221618.583218
Name: Price, Length: 50000, dtype: float64


## Normalize features

In [10]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Define which columns are numeric vs categorical
numeric_features = ['SquareFeet', 'Bedrooms', 'Bathrooms']
categorical_features = ['Neighborhood']

# If x_train/x_test were overwritten by a previous transform, rebuild the raw split
expected_cols = list(x.columns)
needs_reset = (
    (not isinstance(x_train, pd.DataFrame))
    or (not isinstance(x_test, pd.DataFrame))
    or (list(x_train.columns) != expected_cols)
    or (list(x_test.columns) != expected_cols)
)

if needs_reset:
    x_train_raw, x_test_raw, y_train, y_test = train_test_split(
        x, y, test_size=0.2, random_state=42
    )
else:
    x_train_raw, x_test_raw = x_train.copy(), x_test.copy()

# Build a preprocessor that scales numeric features and one-hot encodes categorical ones
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

# Fit on train only; transform both sets
X_train = preprocessor.fit_transform(x_train_raw)
X_test = preprocessor.transform(x_test_raw)

# Keep raw data available too (avoids shape/column mismatch issues)
x_train, x_test = x_train_raw, x_test_raw

In [12]:
from sklearn.linear_model import SGDRegressor

sgda_model = SGDRegressor(loss='squared_error', max_iter=1000, learning_rate = "invscaling", eta0=0.1)
sgda_model.fit(X_train, y_train)
print("Coefficients:", sgda_model.coef_)
print("Intercept:", sgda_model.intercept_)

Coefficients: [ 5.37649624e+04  4.75795907e+03 -1.69130443e-01  4.09832898e+04
  4.15112253e+04  4.36266618e+04]
Intercept: [180577.99893767]


loss ="square_error-> minimising MSE
eta0 -> intial learing rate 
max_iter - number of iteration
learning_rate= "invscaling"
        Early iteration:large steps -> faster learning
        later iiteration :smaller steps -> more stable convergence
        Helps avoid overshooting the minimum of the cost function

In [ ]:
y_pred = sgda_model.predict(X_test)

NameError: name 'sgd_model' is not defined